# PyField Cl₂ walkthrough — `qm-relax` → `make-scan` → `qm-prep` → SA

End-to-end ReaxFF refit driven entirely by QM. The user only types a *rough* geometry and a perturbation grid; PyField does the rest:

1. **`qm-relax`** — PySCF geom-opt finds the equilibrium Cl–Cl bond length (the user's input was off by ~7%).
2. **`make-scan`** — expand the `scans:` block into N perturbed structures + matching `single_point` sims + matching `energy_combination` targets, dump xyz snapshots.
3. **Visualise** the scan inline with `pyfield.viz.animate_xyz_dir` so you can confirm the perturbations look the way you intended.
4. **`qm-prep`** — PySCF single-points fill every `target: { from: dft }`.
5. **SA refit** with before/after `cost_breakdown` so you can see exactly which residuals shrunk.
6. **Reproducibility** — same seed + same QM cache → same FINAL cost.

This notebook is also a regression test (`pytest examples/` re-executes every cell). Requires `pip install -e .[dev]` (pulls `pyscf`, `geometric`, `matplotlib`, `nbmake`) and a working LAMMPS (`pip install 'lammps[mpi]'`).

## 1. Load the rough config

`tests/cl2_scan.yaml` declares a `qm:` block (PySCF / B3LYP / def2-SVP), one Cl₂ structure with a deliberately-off bond length (±1.10 Å vs the real ~1.028 Å) flagged `qm_relax: true`, an empty `simulations:` / `targets:`, and a `scans:` block with five bond stretches. `qm-relax` and `make-scan` will populate everything else.

In [1]:
import os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
os.chdir(ROOT)

from pyfield.config.loader import load_yaml

cfg = load_yaml('tests/cl2_scan.yaml')
print(f'qm:          {cfg.qm.code}/{cfg.qm.functional}/{cfg.qm.basis}')
print(f'structures:  {len(cfg.structures)}  (qm_relax flagged: {[n for n, s in cfg.structures.items() if s.qm_relax]})')
print(f'simulations: {len(cfg.simulations)}  (will be populated by make-scan)')
print(f'targets:     {len(cfg.targets)}  (will be populated by make-scan + qm-prep)')
print(f'scans:       {len(cfg.scans)}  ({cfg.scans[0].type}, N={len(cfg.scans[0].values)})')

qm:          pyscf/b3lyp/def2-svp
structures:  1  (qm_relax flagged: ['Cl2_Opt'])
simulations: 0  (will be populated by make-scan)
targets:     0  (will be populated by make-scan + qm-prep)
scans:       1  (bond_stretch, N=5)


## 2. `qm-relax` — find the equilibrium geometry

PySCF runs a geometric-solver optimisation on every `qm_relax: true` structure and writes the relaxed coordinates back into `atoms:` of a fresh YAML. The flag is dropped on output. Typical wall-clock for Cl₂ / B3LYP / def2-SVP is ~7 s on a modern laptop.

In [2]:
import importlib.util
import shutil
from pathlib import Path

assert importlib.util.find_spec('pyscf') and importlib.util.find_spec('geometric'), (
    'this notebook requires `pip install -e .[dev]` (pulls pyscf + geometric)')

# Wipe the QM cache so this run is reproducible from a clean state.
shutil.rmtree('tests/runs/qm_cache', ignore_errors=True)

from pyfield.qm.prep import cfg_to_yaml, relax_structures

relaxed_cfg, journal = relax_structures(cfg)
for action, hit, key in journal:
    tag = '[cache hit ]' if hit else '[running   ]'
    print(f'  {tag} {action}   ({key})')

relaxed_path = Path('tests/cl2_scan.relaxed.yaml')
relaxed_path.write_text(cfg_to_yaml(relaxed_cfg))

ref = relaxed_cfg.structures['Cl2_Opt']
bond = abs(ref.atoms[1].z - ref.atoms[0].z)
print()
print(f'relaxed YAML written to {relaxed_path}')
print(f'Cl–Cl bond after relax: {bond:.4f} Å  (input was 2.20 Å)')

geometric-optimize called with the following command line:
/home/ubuntu/env-310/lib/python3.10/site-packages/ipykernel_launcher.py --f=/run/user/1000/jupyter/runtime/kernel-v3c4ec446c1fed7d1abd3dd31cc432b7c03b7db89c.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              **

  [running   ] relax Cl2_Opt   (a41384f3b12aba88)

relaxed YAML written to tests/cl2_scan.relaxed.yaml
Cl–Cl bond after relax: 2.0559 Å  (input was 2.20 Å)


## 3. `make-scan` — stamp out perturbed structures

Each entry in `scans:` expands to N structures + N `single_point` sims + N `energy_combination` targets (`Scan_i − Cl2_Opt`, with `target: { from: dft }` waiting for `qm-prep`). The xyz snapshots get written to `--xyz-dir` so you can spot-check them in OVITO — or, better, animate them inline (next cell).

In [3]:
from pyfield.scans import expand_scans

scanned_cfg, summary = expand_scans(relaxed_cfg, xyz_dir=Path('tests/runs/scan_xyz'))
for line in summary:
    print('  ', line)

scanned_path = Path('tests/cl2_scan.scanned.yaml')
scanned_path.write_text(cfg_to_yaml(scanned_cfg))
print()
print(f'scanned YAML written to {scanned_path}')
print(f'structures: {len(scanned_cfg.structures)}  (1 reference + 5 scan points)')
print(f'simulations: {len(scanned_cfg.simulations)}  (1 reference single_point + 5 scan single_points)')
print(f'targets: {len(scanned_cfg.targets)}  (5 energy_combination, all `from: dft`)')

   bond_stretch        ref=Cl2_Opt    prefix=Cl2_d        N=5

scanned YAML written to tests/cl2_scan.scanned.yaml
structures: 6  (1 reference + 5 scan points)
simulations: 6  (1 reference single_point + 5 scan single_points)
targets: 5  (5 energy_combination, all `from: dft`)


## 4. Visualise the scan inline

`animate_xyz_dir` reads every `Cl2_d_*.xyz` in the dump directory, builds a 3D scatter animation (atoms coloured by element), and embeds the play/slider widget directly in the notebook. Use this to confirm the perturbations are doing what you intended *before* paying for QM.

In [4]:
import matplotlib
matplotlib.use('Agg')   # so the notebook test runs in headless CI
from pyfield.viz import animate_xyz_dir

animate_xyz_dir('tests/runs/scan_xyz', pattern='Cl2_d_*.xyz', interval_ms=400,
                title='Cl₂ bond_stretch scan')

Animation.save using <class 'matplotlib.animation.HTMLWriter'>

## 5. `qm-prep` — fill the placeholders

`populate_qm` walks every `from: dft` slot, runs the matching PySCF single-point, and emits the populated YAML. Six jobs total (1 reference + 5 scan points) at B3LYP / def2-SVP — ~30 s on a laptop. Re-running is free thanks to the content-keyed cache.

In [ ]:
from pyfield.qm.prep import populate_qm

populated, journal = populate_qm(scanned_cfg)
for action, hit, key in journal:
    tag = '[cache hit ]' if hit else '[running   ]'
    print(f'  {tag} {action}   ({key})')

populated_path = Path('tests/cl2_scan.populated.yaml')
populated_path.write_text(cfg_to_yaml(populated))

print()
print('DFT ΔE targets (populated):')
for i, (sim, tgt) in enumerate(zip(
        ['Cl2_d_0_sp', 'Cl2_d_1_sp', 'Cl2_d_2_sp', 'Cl2_d_3_sp', 'Cl2_d_4_sp'],
        populated.targets)):
    bond = scanned_cfg.scans  # already None after expand; pull values from input config
    print(f'  {sim:<14}  ΔE = {tgt.__pydantic_extra__["target"]:>10.4f} kcal/mol')

## 6. SA refit with before/after `cost_breakdown`

`cost_breakdown` runs every required LAMMPS simulation once and reports per-structure energies + per-target residuals. The total residual *is* the SA cost, so the breakdown explains the headline number. We print it before and after SA so you can see which targets the optimiser actually fit.

In [ ]:
from pyfield.io.lammps import preload_libmpi
preload_libmpi()
from pyfield.optimizers.sa import run_sa
from pyfield.diagnostics import cost_breakdown

print('=' * 70)
print('BEFORE SA — initial ReaxFF vs B3LYP targets')
print('=' * 70)
cost_breakdown(populated).print_table()

result = run_sa(populated)

print()
print('=' * 70)
print(f'AFTER SA  — best FF written to {result.best_ffield_path}')
print('=' * 70)
cost_breakdown(populated, ffield_path=result.best_ffield_path).print_table()

print()
print(f'SA final cost: {result.final_cost}')
print(f'cost trace length: {len(result.cost_trace)}')

## 7. Cost trace

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(result.cost_trace, marker='.')
ax.set_xlabel('iteration')
ax.set_ylabel('cost')
ax.set_title('Cl₂ ReaxFF refit — cost trace')
fig.tight_layout()
fig.savefig('examples/_cl2_cost_trace.png', dpi=80)

## 8. Reproducibility — every step is content-keyed

Re-running `qm-relax` + `make-scan` + `qm-prep` on the same input must be a 100% cache-hit no-op (same atoms, same QM settings, same op → same SHA-256 → same cache entry). With `optimizer.seed` fixed, SA on the populated config also gives a bit-identical cost.

In [ ]:
cfg2 = load_yaml('tests/cl2_scan.yaml')
relaxed2, jr2 = relax_structures(cfg2)
scanned2, _ = expand_scans(relaxed2)
populated2, jp2 = populate_qm(scanned2)

assert all(hit for _, hit, _ in jr2), 'qm-relax should be all cache hits'
assert all(hit for _, hit, _ in jp2), 'qm-prep should be all cache hits'

for t1, t2 in zip(populated.targets, populated2.targets):
    assert t1.__pydantic_extra__['target'] == t2.__pydantic_extra__['target']

result2 = run_sa(populated2)
assert result.final_cost == result2.final_cost, (result.final_cost, result2.final_cost)

print(f'qm-relax re-run: {sum(1 for _, h, _ in jr2 if h)}/{len(jr2)} cache hits')
print(f'qm-prep  re-run: {sum(1 for _, h, _ in jp2 if h)}/{len(jp2)} cache hits')
print(f'SA re-run cost:  {result2.final_cost}  (matches first run)')
print('OK — the entire qm-relax + make-scan + qm-prep + SA chain is bit-reproducible.')